# 05 - Amendment Override Demo

This notebook demonstrates **amendment-aware retrieval** — the mechanism that ensures the latest version of a clause is returned instead of stale base contract terms.

**What you'll see:**

1. **The problem** — standard similarity search returns outdated base contract terms
2. **The solution** — `AmendmentAwareRetriever` deduplicates by version and returns current terms
3. **Group-by visualization** — how chunks are grouped and pruned
4. **Multi-version history** — retrieving all versions chronologically
5. **Edge cases** — contracts without amendments, cross-contract queries

**Prerequisites:** Run `01_generate_data.ipynb` and `02_ingestion.ipynb` first (data must be ingested into Pinecone).

In [1]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path('../.env'))

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

settings = get_settings()
init_observability(settings)

## 1. The Problem: Standard Similarity Search Returns Stale Terms

Let's query Pinecone **without** amendment-aware deduplication — using a plain vector retriever. We'll ask about the ITSVC-001 hourly rate, which has 3 versions:

| Version | Source | Rate | Effective Date |
|---------|--------|------|----------------|
| v1 | Base contract | $250/hr | 2025-01-15 |
| v2 | Amendment 1 | $285/hr | 2025-07-01 |
| v3 | Amendment 2 | $310/hr | 2026-01-01 |

With standard similarity search, the base contract chunk often ranks highest because it has more surrounding context.

In [2]:
from llama_index.core import VectorStoreIndex
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from contract_lens.ingestion.pipeline import build_embedding_model, build_pinecone_vector_store

# Connect to Pinecone
vector_store = build_pinecone_vector_store(settings)
embed_model = build_embedding_model(settings)
index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed_model)

# Standard retriever — NO amendment deduplication
filters = MetadataFilters(filters=[
    MetadataFilter(key="contract_id", value="ITSVC-001", operator=FilterOperator.EQ),
])
plain_retriever = index.as_retriever(similarity_top_k=8, filters=filters)

QUERY = "What is the current hourly rate?"
plain_results = plain_retriever.retrieve(QUERY)

print(f"Standard retriever returned {len(plain_results)} chunks:\n")
print(f"{'Rank':<5} {'Score':<8} {'Version':<8} {'Source':<12} {'Eff. Date':<12} {'Section':<12} First 80 chars")
print("-" * 120)
for i, node in enumerate(plain_results):
    m = node.metadata or {}
    print(f"{i+1:<5} {node.score or 0:.4f}   v{m.get('version','?'):<6} {m.get('source_type','?'):<12} {m.get('effective_date','?'):<12} {m.get('section_type','?'):<12} {node.text[:80]}...")

Standard retriever returned 8 chunks:

Rank  Score    Version  Source       Eff. Date    Section      First 80 chars
------------------------------------------------------------------------------------------------------------------------
1     0.4512   v2      amendment    2025-07-01   annex        ## 1. Revised Pricing (replaces Annex A)
Due to increased market rates and expan...
2     0.4467   v3      amendment    2026-01-01   annex        ## 1. Revised Pricing (replaces previous Annex A)
The parties agree to the follo...
3     0.4411   v1      base         2025-01-15   annex        # ANNEX A - Pricing Schedule
The following rates apply to services rendered unde...
4     0.3138   v3      amendment    2026-01-01   general      ## 2. New Service Tier (new clause 1.4)
A new clause 1.4 is added: 'The Provider...
5     0.2660   v1      base         2025-01-15   payment      ## 3. Payment Terms
3.1
The Client shall pay the Provider in accordance with the...
6     0.2424   v3      amendment

Notice how chunks from **v1 (base)** may appear above **v3 (latest amendment)** in the ranking. The LLM would see stale `$250/hr` terms before the current `$310/hr`.

## 2. The Solution: AmendmentAwareRetriever

Now let's use `AmendmentAwareRetriever` which:
1. **Over-fetches** 3x more chunks
2. **Groups** by `(contract_id, section_type, clause_number)`
3. **Deduplicates** — keeps only the latest version per group
4. **Re-scores** with a small version boost

In [3]:
from contract_lens.retrieval.amendment_retriever import AmendmentAwareRetriever

# Amendment-aware retriever — WITH deduplication
amendment_retriever = AmendmentAwareRetriever(
    index=index,
    top_k=8,
    filters=filters,
)

amendment_results = amendment_retriever.retrieve(QUERY)

print(f"AmendmentAwareRetriever returned {len(amendment_results)} chunks:\n")
print(f"{'Rank':<5} {'Score':<8} {'Version':<8} {'Source':<12} {'Eff. Date':<12} {'Section':<12} First 80 chars")
print("-" * 120)
for i, node in enumerate(amendment_results):
    m = node.metadata or {}
    print(f"{i+1:<5} {node.score or 0:.4f}   v{m.get('version','?'):<6} {m.get('source_type','?'):<12} {m.get('effective_date','?'):<12} {m.get('section_type','?'):<12} {node.text[:80]}...")

AmendmentAwareRetriever returned 8 chunks:

Rank  Score    Version  Source       Eff. Date    Section      First 80 chars
------------------------------------------------------------------------------------------------------------------------
1     0.4967   v3      amendment    2026-01-01   annex        ## 1. Revised Pricing (replaces previous Annex A)
The parties agree to the follo...
2     0.3638   v3      amendment    2026-01-01   general      ## 2. New Service Tier (new clause 1.4)
A new clause 1.4 is added: 'The Provider...
3     0.2924   v3      amendment    2026-01-01   payment      ## 3. Payment Terms (amends clause 3.2)
Clause 3.2 is amended to read: 'Invoices...
4     0.2827   v1      base         2025-01-15   payment      ## 3. Payment Terms
3.1
The Client shall pay the Provider in accordance with the...
5     0.2362   v3      amendment    2026-01-01   general      ## 4. Unchanged Terms
All other terms and conditions of the Agreement, as previo...
6     0.2337   v3      amen

The stale v1/v2 chunks for the same section are **pruned**. Only the latest version of each clause remains.

## 3. Group-by Visualization

Let's see exactly how chunks are grouped and which get pruned during deduplication.

In [4]:
from collections import defaultdict
from contract_lens.retrieval.amendment_retriever import _grouping_key, _version_sort_key

# Over-fetch the same query with a large K to see all candidate chunks
large_retriever = index.as_retriever(similarity_top_k=24, filters=filters)
all_candidates = large_retriever.retrieve(QUERY)

# Group them
groups = defaultdict(list)
for node in all_candidates:
    key = _grouping_key(node)
    groups[key].append(node)

print(f"Over-fetched {len(all_candidates)} chunks, grouped into {len(groups)} groups:\n")
for key, group_nodes in sorted(groups.items()):
    contract_id, section_type, clause_num = key
    max_ver = max(_version_sort_key(n)[0] for n in group_nodes)
    print(f"Group: ({contract_id}, {section_type}, '{clause_num}') — {len(group_nodes)} chunk(s)")
    for node in sorted(group_nodes, key=lambda n: _version_sort_key(n), reverse=True):
        ver, date = _version_sort_key(node)
        kept = "KEEP" if ver == max_ver else "PRUNE"
        print(f"  [{kept}] v{ver} ({date}) score={node.score or 0:.4f}  {node.text[:60]}...")
    print()

Over-fetched 17 chunks, grouped into 8 groups:

Group: (ITSVC-001, annex, '') — 3 chunk(s)
  [KEEP] v3 (2026-01-01) score=0.4469  ## 1. Revised Pricing (replaces previous Annex A)
The partie...
  [PRUNE] v2 (2025-07-01) score=0.4514  ## 1. Revised Pricing (replaces Annex A)
Due to increased ma...
  [PRUNE] v1 (2025-01-15) score=0.4413  # ANNEX A - Pricing Schedule
The following rates apply to se...

Group: (ITSVC-001, confidentiality, '4.1') — 1 chunk(s)
  [KEEP] v1 (2025-01-15) score=0.1070  ## 4. Confidentiality
4.1
Both parties agree to maintain the...

Group: (ITSVC-001, general, '') — 8 chunk(s)
  [KEEP] v3 (2026-01-01) score=0.3139  ## 2. New Service Tier (new clause 1.4)
A new clause 1.4 is ...
  [KEEP] v3 (2026-01-01) score=0.1863  ## 4. Unchanged Terms
All other terms and conditions of the ...
  [KEEP] v3 (2026-01-01) score=0.1839  # AMENDMENT NO. 2


## TO IT SERVICE AGREEMENT (ITSVC-001)
C...
  [PRUNE] v2 (2025-07-01) score=0.1928  # AMENDMENT NO. 1


## TO IT SERVICE AGREEM

## 4. Full Query Engine Comparison: Before vs. After

Let's compare the **full pipeline** (retrieval + LLM synthesis) — the actual answer the user would see.

In [5]:
from contract_lens.retrieval.query_engine import query_contracts

# Test case 1: ITSVC-001 hourly rate (expected: $310/hr from v3)
print("=" * 80)
print("Q: What is the current hourly rate for ITSVC-001?")
print("Expected: $310/hr (from Amendment 2, v3, effective 2026-01-01)")
print("=" * 80)
answer = query_contracts(settings, "What is the current hourly rate?", contract_id="ITSVC-001")
print(answer)

Q: What is the current hourly rate for ITSVC-001?
Expected: $310/hr (from Amendment 2, v3, effective 2026-01-01)


The current hourly rates (effective January 1, 2026) are:

- Senior Architect: **$310/hour**
- Cloud Engineer: **$260/hour**
- DevOps Specialist: **$235/hour**
- Project Manager: **$195/hour**
- QA Engineer: **$175/hour**
- AI/ML Engineer: **$320/hour**


In [6]:
# Test case 2: SLA-001 P1 response time (expected: 5 min from amendment)
print("=" * 80)
print("Q: What is the P1 incident response time in SLA-001?")
print("Expected: 5 minutes (from Amendment, v2, effective 2025-10-01)")
print("=" * 80)
answer = query_contracts(settings, "What is the P1 incident response time?", contract_id="SLA-001")
print(answer)

Q: What is the P1 incident response time in SLA-001?
Expected: 5 minutes (from Amendment, v2, effective 2025-10-01)


The P1 (Critical) incident response time is **5 minutes**.


In [7]:
# Test case 3: LEASE-001 monthly rent (expected: 28,500 PLN from amendment)
print("=" * 80)
print("Q: Jaki jest miesięczny czynsz w umowie LEASE-001?")
print("Expected: 28,500 PLN (from Aneks, v2, effective 2025-09-01)")
print("=" * 80)
answer = query_contracts(settings, "Jaki jest miesięczny czynsz?", contract_id="LEASE-001", language="pl")
print(answer)

Q: Jaki jest miesięczny czynsz w umowie LEASE-001?
Expected: 28,500 PLN (from Aneks, v2, effective 2025-09-01)


Miesięczny czynsz najmu wynosi **28 500 PLN netto** (plus **VAT 23%**).


## 5. Multi-Version History

When asked about **history of changes**, we want ALL versions — not just the latest. The amendment-aware retriever deduplicates per group, but a "history" query naturally matches chunks from different versions differently, so multiple versions can survive in different groups or be explicitly requested.

In [8]:
# Test case 4: Rate change history (should list all versions chronologically)
print("=" * 80)
print("Q: Show me the history of hourly rate changes for ITSVC-001")
print("Expected: $250 (v1, 2025-01-15) → $285 (v2, 2025-07-01) → $310 (v3, 2026-01-01)")
print("=" * 80)
answer = query_contracts(
    settings,
    "Show me the complete history of hourly rate changes, listing all versions chronologically",
    contract_id="ITSVC-001",
)
print(answer)

Q: Show me the history of hourly rate changes for ITSVC-001
Expected: $250 (v1, 2025-01-15) → $285 (v2, 2025-07-01) → $310 (v3, 2026-01-01)


Hourly rate history by version (chronological)

1) Version 1 (Base Contract) — Effective 2025-01-15  
- Hourly rates: Not provided in the available text (rates are referenced as being in “Annex A,” but the Annex A rate table is not included here).

2) Version 3 (Amendment No. 2) — Effective 2026-01-01 (replaces previous Annex A)  
- Senior Architect: $310/hour  
- Cloud Engineer: $260/hour  
- DevOps Specialist: $235/hour  
- Project Manager: $195/hour  
- QA Engineer: $175/hour  
- AI/ML Engineer: $320/hour


## 6. Edge Cases

### 6a. Contract without amendments (NDA-001)

NDA-001 has no amendments. Base contract terms must NOT be pruned by deduplication.

In [9]:
# Test case 5: NDA-001 (no amendments — base terms should survive dedup)
print("=" * 80)
print("Q: What are the confidentiality obligations in NDA-001?")
print("Expected: Base contract terms (v1) — dedup should NOT remove them")
print("=" * 80)
answer = query_contracts(
    settings,
    "What are the confidentiality obligations and non-disclosure terms?",
    contract_id="NDA-001",
)
print(answer)

Q: What are the confidentiality obligations in NDA-001?
Expected: Base contract terms (v1) — dedup should NOT remove them


- **Confidentiality obligations (Clause 3.1):** Each receiving party must (a) hold the other party’s Confidential Information in strict confidence, (b) not disclose it to any third party without the disclosing party’s prior written consent, and (c) use it solely for the stated Purpose (exploring a potential business relationship concerning joint development of AI-powered analytics solutions).

- **Access restrictions (Clause 3.2):** Each party must limit access to Confidential Information to employees and contractors who have a need to know and who are bound by confidentiality obligations.

- **What counts as “Confidential Information” (Clause 2.1) and exclusions (Clause 2.2):** It includes any non-public information disclosed by either party (e.g., technical data, trade secrets, business plans, customer lists, financial information, source code, algorithms, product roadmaps). It excludes information that is publicly available, was known prior to disclosure, is independently developed,

### 6b. Cross-contract query

Query without `contract_id` filter — results should span multiple contracts, each showing their latest version.

In [10]:
# Test case 6: Cross-contract — latest pricing tables
print("=" * 80)
print("Q: What pricing tables exist across all contracts?")
print("Expected: Latest pricing from each contract (ITSVC-001 v3, SLA-001 v2, etc.)")
print("=" * 80)
answer = query_contracts(
    settings,
    "Show me all pricing tables and rate schedules across all contracts",
    has_table=True,
)
print(answer)

Q: What pricing tables exist across all contracts?
Expected: Latest pricing from each contract (ITSVC-001 v3, SLA-001 v2, etc.)


### ITSVC-001 — Revised Pricing (effective 2026-01-01; replaces previous Annex A)

| Service Category | Rate (USD/hour) | Min Hours/Month | Monthly Cap |
|---|---:|---:|---:|
| Senior Architect | $310 | 40 | $15,000 |
| Cloud Engineer | $260 | 80 | $23,000 |
| DevOps Specialist | $235 | 60 | $16,000 |
| Project Manager | $195 | 20 | $5,000 |
| QA Engineer | $175 | 40 | $8,500 |
| AI/ML Engineer | $320 | 40 | $15,500 |

---

### EMP-001 — Wynagrodzenie i świadczenia (Annex 1; effective 2025-05-01)

| Składnik | Kwota/Opis | Częstotliwość |
|---|---|---|
| Wynagrodzenie zasadnicze | 22 000 PLN brutto | Miesięczne |
| Premia uznaniowa | Do 20% wynagrodzenia zasadniczego | Kwartalne |
| Premia roczna | Do 2 wynagrodzeń zasadniczych | Roczne |
| Pakiet medyczny (LuxMed) | Wariant rodzinny | Miesięczne |
| Karta sportowa (MultiSport) | Wariant Plus | Miesięczne |
| Budżet szkoleniowy | 10 000 PLN rocznie | Roczne |
| Sprzęt służbowy | MacBook Pro 16" + monitor 4K | Jednorazowo |
| Dodatek za

## Summary

| Feature | Standard Retriever | AmendmentAwareRetriever |
|---------|-------------------|------------------------|
| Fetches | top_k chunks by similarity | 3x top_k, then deduplicates |
| Version awareness | None — relies on LLM prompt | Groups by clause, keeps latest version only |
| Stale terms in context | Likely (base contracts rank higher) | Removed at retrieval level |
| Top-K slot efficiency | Wasted on duplicate clause versions | One slot per clause (latest) |
| Config | `similarity_top_k` | `top_k`, `fetch_k`, `version_boost` |

The `AmendmentAwareRetriever` is integrated into the default query engine — all agent queries automatically benefit.

See [docs/amendment-override.md](../docs/amendment-override.md) for the full algorithm and configuration.